# U-Values and Thermal Calculations

This notebook calculates thermal conductivity (lambda) values, R-values, and U-values for building materials, with a focus on External Wall Insulation (EWI) payback analysis.

## Setup: Import Libraries

In [1]:
# %pip install pint
# I used to use the above, which forces load of the pint library.
# (you need this library, but better to install it via 
# `python -m ipykernel install --user --name=Part-L --display-name "Part-L"` 
# you need to have the right libraries installed in the active environment before doing the ipykernel install. 
# best to manage the environments using conda, in my limited experience.
import pint

ureg = pint.UnitRegistry()
# ureg.define('gbp = []')  # Define GBP as a dimensionless currency unit
ureg.define('gbp = [currency]') 
ureg.gbp.symbol = '£'


## Material Properties: Thermal Conductivity (Lambda)

Units: W/mK (watts per metre-kelvin)

In [2]:
LAMBDA = {    
    "xps": 0.04,  # Polyisocyanurate or extruded polystyrene    
    "brick": 0.77,  # Medium density clay brick (~1800 kg/m³)
    "clay_brick_lightweight": 0.5,  # Lightweight clay brick (< 1700 kg/m³)    
    "plasterboard": 0.25,  # Standard plasterboard (gypsum)
    "plaster": 0.77 # assumed
    # add a few more here when everything else works
}

## Wall Construction Definition

Thicknesses in meters

## R and U from thickness and conductivity

In [3]:


def lambda_val(material):
    """Get thermal conductivity with units."""
    return LAMBDA[material] * ureg.W / (ureg.m * ureg.kelvin) 

def R_value(element, thickness):
    """Calculate R-value (thermal resistance) for a wall element."""
    return  thickness/ lambda_val(element)

def U_value(wall):
    """
    The U value of the wall, in SI units.
    """
    units_of_U = ureg.watt / ureg.meter ** 2 / ureg.K
    tot_R = 0.0 / units_of_U
    for i in wall:
        R = R_value(i,wall[i] * ureg.meters)
        tot_R += R
    return 1./tot_R
    


In [4]:
assert lambda_val('plaster') == 0.77 * ureg.W / ureg.m / ureg.K

In [5]:
lambda_val('xps')


<Quantity(0.04, 'watt / meter / kelvin')>

## Calculate Total R-Value and U-Value for Wall

In [6]:

u_with_EWI = U_value(
    {
    "plasterboard": 0.013,  # Internal plaster finish
    "brick": 0.220,  # Main structural brick layer
    "xps": 0.0700,})  # Internal insulation board

print(f"u_with_EWI: {u_with_EWI:.2fP~}")

u_with_EWI: 0.48 W/K/m²


In [7]:

u_without_EWI = U_value(
    {
    "plaster": 0.025,  # Internal plaster finish
    "brick": 0.2200,  # Main structural brick layer
    }) 

print(f"u_without_EWI: {u_without_EWI:.2fP~}")



u_without_EWI: 3.14 W/K/m²


## External Wall Insulation (EWI) Analysis

### Energy and Cost Calculations

In [31]:
def cost_saving(nom_gas_price, nom_degree_days, install_cost_nom, before, after):
    """
    nom_gas_price: as on your bill, but in £ (e.g. 3p/kWh = 0.03, but with units)
    nom_degree_days: e.g. 2000, usual definition.
    install_cost: cost psm to install EWI/IWI    
    before: U-value before adding insulation
    after: U-value after adding insulation

    returns [annual cost of extra energy, a payback period, in years]
    """
    degree_days = nom_degree_days * ureg.K * 60*60*24 * ureg.s   # get in SI units
    # degree_days = nom_degree_days * ureg.K * 60*60*24 * ureg.s  # Assumed for Knebworth (base 15°C)
    gas_price = nom_gas_price*ureg.gbp / ((1000 * ureg.W) * (60*60*ureg.s))   # gbp/kWh (2021 pre-crisis price)
    # print(f"gas price {gas_price:.2eP~}")
    delta_u = before - after
    # print(f"Change in U-value from EWI: {delta_u:.4fP~}")
    saving = gas_price * delta_u # per unit time
    # print(f"saving: {saving:.6eP~}")
    saving_pd = saving*60*60*24*ureg.s
    # print(f"saving pd: {saving_pd:.2eP~}")

    year_cost_psm = degree_days  * gas_price * delta_u

    # print(f"Annual cost saving: {year_cost_psm:.2fP~}")
    install_cost = install_cost_nom * ureg.gbp / (ureg.meter**2)  # gbp/m²
    # print(f"Installation cost: {installation_cost:.0fP~}")
    # print(f"Annual saving per m²: {year_cost_psm:.2fP~}")
    # print(f"payback: {installation_cost/saving/degree_days:.2fP~} years")
    return [year_cost_psm, installation_cost/saving/degree_days]


In [32]:

nom_degree_days = 2009.3
nom_gas_price = 0.03
u_without_EWI - u_with_EWI
gas = 0.03
install = 200
[annual_saving, payback] = cost_saving(gas, 2009.3, install, u_without_EWI, u_with_EWI)

print(f"At {gas*100}p/kWh, and an install cost of £{install}/m^2, annual saving psm is {annual_saving:.2fP~}, payback period: {payback:.2fP~}") 

gas = 0.06
install = 150

[annual_saving, payback] = cost_saving(0.06, 2009.3, 150, u_without_EWI, u_with_EWI)
print(f"At {gas*100}p/kWh, and an install cost of £{install}/m^2, annual saving psm is {annual_saving:.2fP~}, payback period: {payback:.2fP~}") 



At 3.0p/kWh, and an install cost of £200/m^2, annual saving psm is 3.85 gbp/m², payback period: 51.90
At 6.0p/kWh, and an install cost of £150/m^2, annual saving psm is 7.71 gbp/m², payback period: 25.95


### Payback Calculation